# 07.03 · Vector databases compared

**Chapter:** Embeddings & Vector Search · **Milestone:** M3

## Goals
- [ ] Load the same data into four stores and run the same filtered query in each
- [ ] Understand what a vector DB adds over a FAISS index: persistence, metadata filters, updates/deletes, APIs, scaling
- [ ] Pick one for a project

## Resources
- [Chroma docs](https://docs.trychroma.com/) · [Qdrant docs](https://qdrant.tech/documentation/) · [LanceDB docs](https://lancedb.github.io/lancedb/) — free
- [pgvector](https://github.com/pgvector/pgvector) · [sqlite-vec](https://alexgarcia.xyz/sqlite-vec/) — free
- [Superlinked – Vector DB comparison table](https://superlinked.com/vector-db-comparison) — free

| Store | Runs as | Good for |
|---|---|---|
| **Chroma** | embedded (in-process) or server | prototyping, small apps; simplest API |
| **LanceDB** | embedded, files on disk / S3 (Lance columnar format) | larger-than-RAM data, multimodal, analytics + search |
| **Qdrant** | local mode (in-process) or server / cloud | production self-hosting, rich filtering, quantisation |
| **Weaviate**, **Milvus** | server / cloud | large scale, built-in hybrid search, multi-tenancy |
| **pgvector** | Postgres extension | you already run Postgres: one DB, SQL joins, transactions |
| **sqlite-vec** | SQLite extension | tiny/edge/local apps, the SQLite of vector search |
| **Pinecone**, Turbopuffer, Zilliz, Qdrant/Weaviate Cloud | managed | no ops, serverless pricing |
| **Elasticsearch / OpenSearch** | server | you already run them; strong lexical + vector hybrid |

Everything here runs **in-process, no Docker**. Server versions use the same client APIs.

In [ ]:
import shutil
import time

import numpy as np

from aieng import DATA, rag

docs, X = rag.corpus(), rag.index()
# Synthetic metadata so we can filter: passage length bucket + a fake "source" field
meta = [{"id": i, "length": "long" if len(d) > 600 else "short", "source": ["wiki-a", "wiki-b", "wiki-c"][i % 3]} for i, d in enumerate(docs)]
QUERY = "When was Abraham Lincoln born?"
qvec = rag.embed([QUERY], query=True)[0]
DB_DIR = DATA / "vectordbs"
shutil.rmtree(DB_DIR, ignore_errors=True); DB_DIR.mkdir()


def show(name, rows, secs):
    print(f"--- {name} ({secs * 1000:.1f} ms)")
    for pid, score, text in rows:
        print(f"  {score:.3f} [{pid}] {text[:80]}")

## 1. Chroma
Collections hold ids, embeddings, documents and metadata. Chroma can embed for you; here we pass our own vectors so all stores match.

In [ ]:
import chromadb

chroma = chromadb.PersistentClient(path=str(DB_DIR / "chroma"))
col = chroma.create_collection("wiki", metadata={"hnsw:space": "cosine"})
for s in range(0, len(docs), 1000):  # add in batches
    col.add(ids=[str(i) for i in range(s, min(s + 1000, len(docs)))], embeddings=X[s : s + 1000],
            documents=docs[s : s + 1000], metadatas=meta[s : s + 1000])

start = time.perf_counter()
r = col.query(query_embeddings=[qvec], n_results=3, where={"length": "long"})
show("Chroma, where length=long", [(int(i), 1 - dist, t) for i, dist, t in zip(r["ids"][0], r["distances"][0], r["documents"][0])], time.perf_counter() - start)

## 2. Qdrant (local mode)
Same client works against `QdrantClient(url="http://localhost:6333")` or Qdrant Cloud — swap one line.

In [ ]:
from qdrant_client import QdrantClient, models

qd = QdrantClient(path=str(DB_DIR / "qdrant"))
qd.create_collection("wiki", vectors_config=models.VectorParams(size=X.shape[1], distance=models.Distance.COSINE))
qd.upload_points("wiki", [models.PointStruct(id=i, vector=X[i].tolist(), payload={**meta[i], "text": docs[i]}) for i in range(len(docs))])

start = time.perf_counter()
res = qd.query_points("wiki", query=qvec.tolist(), limit=3, query_filter=models.Filter(
    must=[models.FieldCondition(key="length", match=models.MatchValue(value="long"))])).points
show("Qdrant, filter length=long", [(p.id, p.score, p.payload["text"]) for p in res], time.perf_counter() - start)

## 3. LanceDB
Tables are Arrow/Lance files; filters are SQL-like strings. Great when data is big or you want to query it with DuckDB/pandas too.

In [ ]:
import lancedb

ldb = lancedb.connect(str(DB_DIR / "lance"))
table = ldb.create_table("wiki", data=[{"vector": X[i], "text": docs[i], **meta[i]} for i in range(len(docs))])

start = time.perf_counter()
rows = table.search(qvec).metric("cosine").where("length = 'long'").limit(3).to_list()
show("LanceDB, where length = 'long'", [(r["id"], 1 - r["_distance"], r["text"]) for r in rows], time.perf_counter() - start)

(No index built here = exact scan, fine for 3k rows. For millions: `table.create_index(...)` builds IVF-PQ / HNSW.)

## 4. sqlite-vec: vectors inside your SQL database
Same idea as **pgvector** for Postgres: vectors live next to your relational data, filtered and joined with SQL.

In [ ]:
import sqlite3

import sqlite_vec

db = sqlite3.connect(DB_DIR / "wiki.sqlite")
db.enable_load_extension(True); sqlite_vec.load(db); db.enable_load_extension(False)
db.execute(f"CREATE VIRTUAL TABLE vec USING vec0(id INTEGER PRIMARY KEY, embedding float[{X.shape[1]}] distance_metric=cosine, length TEXT)")
db.execute("CREATE TABLE passages(id INTEGER PRIMARY KEY, text TEXT, source TEXT)")
db.executemany("INSERT INTO vec(id, embedding, length) VALUES (?, ?, ?)", [(i, X[i].tobytes(), meta[i]["length"]) for i in range(len(docs))])
db.executemany("INSERT INTO passages VALUES (?, ?, ?)", [(i, docs[i], meta[i]["source"]) for i in range(len(docs))])
db.commit()

start = time.perf_counter()
rows = db.execute("""
    SELECT v.id, v.distance, p.text FROM vec v JOIN passages p ON p.id = v.id
    WHERE v.embedding MATCH ? AND k = 3 AND v.length = 'long' ORDER BY v.distance""", [qvec.tobytes()]).fetchall()
show("sqlite-vec + SQL join", [(i, 1 - dist, t) for i, dist, t in rows], time.perf_counter() - start)

## 5. pgvector and managed services (reference)
Need a server, so they're guarded. `brew install postgresql@17 pgvector` (or a free Supabase/Neon database) gives you pgvector:
```sql
CREATE EXTENSION vector;
CREATE TABLE passages (id int PRIMARY KEY, text text, length text, embedding vector(384));
CREATE INDEX ON passages USING hnsw (embedding vector_cosine_ops);
SELECT id, 1 - (embedding <=> $1) AS score FROM passages WHERE length = 'long' ORDER BY embedding <=> $1 LIMIT 3;
```
Pinecone (free starter tier):
```python
from pinecone import Pinecone            # uv add pinecone
index = Pinecone(api_key=...).Index("wiki")
index.query(vector=qvec.tolist(), top_k=3, filter={"length": {"$eq": "long"}}, include_metadata=True)
```

## 6. Choosing
1. Already on Postgres? **pgvector** until it hurts (often 10M+ vectors with good indexes).
2. Prototype / single process? **Chroma** or **LanceDB** (or just NumPy/FAISS, 07.02).
3. Self-hosted production with heavy filtering? **Qdrant** / **Weaviate** / **Milvus**.
4. No ops at all? A managed service.

What matters more than the DB: chunking (08.02), the embedding model (07.01), hybrid search (07.04) and reranking (07.05).

## Exercises
1. Measure query latency (median of 100 queries from `rag.eval_set()`) and on-disk size for each store.
2. Implement update and delete: change one passage's text + vector and delete another, in Chroma and Qdrant. Verify with a query.
3. Run Qdrant as a server: `brew install qdrant` (or download the binary), start it, and point `QdrantClient(url=...)` at it. Same code?

In [ ]:
# your code here